# DP/SFA SSN accuracy: time-step and frequency sweeps

The companion notebook `DP_generalizedSSN_RLC` showed that the DP SSN components
reproduce the classical circuit. This notebook quantifies the shifted-frequency
accuracy advantage on the same series RLC one-port (R = 10 Ohm, L = 0.01 H,
C = 0.002 F), using a small-step EMT simulation as the reference (the true
value). Accuracy is the error of the steady-state inductor-current phasor,
extracted by a least-squares fit over the second half of each run.

Three sweeps:
- time step, with the source on the carrier (absolute phasor error);
- signal frequency around the carrier, at a fine time step (relative error);
- signal frequency around the carrier, at a large time step (relative error).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dpsimpy
from villas.dataprocessing.readtools import read_timeseries_csv

In [ ]:
R = 10.0
L = 0.01
C = 0.002

f0 = 50.0
vref = 1.0 * np.exp(1j * np.deg2rad(-90.0))

dt_ref = 1e-5
tf = 0.3

log_dir = "logs"

In [ ]:
def run(sim_name, system, domain, comp, attr, dt):
    dpsimpy.Logger.set_log_dir(f"{log_dir}/{sim_name}")
    logger = dpsimpy.Logger(sim_name)
    logger.log_attribute("y", attr, comp)

    sim = dpsimpy.Simulation(sim_name, dpsimpy.LogLevel.off)
    sim.set_system(system)
    sim.set_domain(domain)
    sim.set_time_step(dt)
    sim.set_final_time(tf)
    sim.add_logger(logger)
    sim.run()

    return read_timeseries_csv(
        f"{log_dir}/{sim_name}/{sim_name}.csv", print_status=False
    )["y"]


def steady_phasor(series, w):
    # Steady-state phasor via least-squares fit to cos/sin at w: the metric that
    # exposes frequency warping (a raw-waveform RMSE hides it at large steps).
    t, y = series.time, series.values
    half = slice(len(t) // 2, None)
    ts, ys = t[half], y[half]
    basis = np.column_stack([np.cos(w * ts), np.sin(w * ts), np.ones_like(ts)])
    a, b, _ = np.linalg.lstsq(basis, ys, rcond=None)[0]
    return a - 1j * b


def dp_node(name):
    return dpsimpy.dp.SimNode(name)


def emt_node(name):
    return dpsimpy.emt.SimNode(name, dpsimpy.PhaseType.Single)

In [ ]:
def build_ssn(ph1, node, gnd, src_freq):
    n1 = node("n1")
    vs = ph1.VoltageSource("vs")
    vs.set_parameters(vref, src_freq)
    rlc = ph1.Full_Serial_RLC("rlc")
    rlc.set_parameters(R, L, C)
    vs.connect([gnd, n1])
    rlc.connect([n1, gnd])
    return dpsimpy.SystemTopology(f0, [n1], [vs, rlc]), rlc


def build_rlc(ph1, node, gnd, src_freq):
    n1, n2, n3 = node("n1"), node("n2"), node("n3")
    vs = ph1.VoltageSource("vs")
    vs.set_parameters(vref, src_freq)
    res = ph1.Resistor("r")
    res.set_parameters(R)
    ind = ph1.Inductor("l")
    ind.set_parameters(L)
    cap = ph1.Capacitor("c")
    cap.set_parameters(C)
    vs.connect([gnd, n1])
    res.connect([n1, n2])
    ind.connect([n2, n3])
    cap.connect([n3, gnd])
    return dpsimpy.SystemTopology(f0, [n1, n2, n3], [vs, res, ind, cap]), ind

`reference_phasor` runs a fine-step EMT simulation and returns the
steady-state phasor at the signal frequency (the truth). `emt_ssn_error` and
`dp_ssn_error` run the SSN models at a given step and signal frequency and
return the phasor error against that reference. The DP source frequency is the
offset from the carrier; the DP result is reconstructed with `frequency_shift`
before the phasor is extracted.

In [ ]:
dp_gnd = dpsimpy.dp.SimNode.gnd
emt_gnd = dpsimpy.emt.SimNode.gnd


def reference_phasor(f_sig):
    sys, comp = build_rlc(dpsimpy.emt.ph1, emt_node, emt_gnd, f_sig)
    series = run(f"REF_{f_sig:g}Hz", sys, dpsimpy.Domain.EMT, comp, "i_intf", dt_ref)
    return steady_phasor(series, 2.0 * np.pi * f_sig)


def emt_ssn_error(f_sig, dt, ref):
    sys, comp = build_ssn(dpsimpy.emt.ph1, emt_node, emt_gnd, f_sig)
    series = run(
        f"EMT_SSN_{f_sig:g}Hz_{dt:.0e}", sys, dpsimpy.Domain.EMT, comp, "i_intf", dt
    )
    return np.abs(steady_phasor(series, 2.0 * np.pi * f_sig) - ref)


def dp_ssn_error(f_sig, dt, ref):
    sys, comp = build_ssn(dpsimpy.dp.ph1, dp_node, dp_gnd, f_sig - f0)
    series = run(
        f"DP_SSN_{f_sig:g}Hz_{dt:.0e}", sys, dpsimpy.Domain.DP, comp, "i_intf", dt
    )
    shifted = series.frequency_shift(f0)
    return np.abs(steady_phasor(shifted, 2.0 * np.pi * f_sig) - ref)

## Sweep over time step (source on the carrier)

Trapezoidal integration warps the L and C companion models by
`Psi(w) = tan(w h / 2) / (w h / 2)`, distorting the impedance at the oscillation
frequency. With the source on the carrier the EMT-SSN 50 Hz forced response sits
where `Psi > 1`, so its error grows as the step `h` grows. In DP-SSN the forced
response is the DC of the envelope (`Psi ~ 1`), so the error stays flat.

In [ ]:
ref_carrier = reference_phasor(f0)
steps = [5e-3, 2e-3, 1e-3, 5e-4, 2e-4, 1e-4]

err_emt = [emt_ssn_error(f0, dt, ref_carrier) for dt in steps]
err_dp = [dp_ssn_error(f0, dt, ref_carrier) for dt in steps]

for dt, ee, ed in zip(steps, err_emt, err_dp):
    print(
        f"dt={dt:.0e}  EMT-SSN err={ee:.3e}  DP-SSN err={ed:.3e}  ratio={ee / ed:.0f}"
    )

In [ ]:
plt.figure(figsize=(9, 4))
plt.loglog(steps, err_emt, "o-", label="EMT-SSN")
plt.loglog(steps, err_dp, "s-", label="DP-SSN")
plt.xlabel("time step [s]")
plt.ylabel("steady-state phasor error [A]")
plt.title("Accuracy versus time step (source on the 50 Hz carrier)")
plt.legend()
plt.grid(True, which="both", linestyle=":")
plt.show()

In [ ]:
assert err_dp[0] < err_emt[0] / 100
assert max(err_dp) < 20 * min(err_dp)

## Sweep over signal frequency (fine time step)

The source is now detuned from the carrier and swept on both sides of it. The DP
envelope rotates at the offset `f_sig - f0`, so DP-SSN warps with that offset,
whereas EMT-SSN warps with the absolute signal frequency. The error is reported
relative to the reference phasor, because the RLC response amplitude varies by
orders of magnitude across the band.

At this fine step both methods sit near the floor set by the reference
discretisation (~1e-7 relative), so the point-to-point detail is not physically
meaningful, only the trend is. DP-SSN is most accurate at the carrier. Below
roughly 25-30 Hz the envelope rotates fast enough that EMT-SSN, sitting at a low
absolute frequency, warps less and wins.

In [ ]:
freqs = [10, 20, 25, 30, 35, 40, 45, 50, 55, 60, 70, 90, 120, 160]
refs = [reference_phasor(f) for f in freqs]
carrier = freqs.index(50)

dt_fine = 1e-4
emt_fine = [emt_ssn_error(f, dt_fine, r) / abs(r) for f, r in zip(freqs, refs)]
dp_fine = [dp_ssn_error(f, dt_fine, r) / abs(r) for f, r in zip(freqs, refs)]

for f, ee, ed in zip(freqs, emt_fine, dp_fine):
    print(
        f"f_sig={f:3d} Hz (offset {f - f0:+.0f})  EMT-SSN rel={ee:.3e}  DP-SSN rel={ed:.3e}  ratio={ee / ed:.2f}"
    )

In [ ]:
plt.figure(figsize=(9, 4))
plt.semilogy(freqs, emt_fine, "o-", label="EMT-SSN")
plt.semilogy(freqs, dp_fine, "s-", label="DP-SSN")
plt.axvline(f0, color="0.7", linestyle=":", label="carrier")
plt.xlabel("signal frequency [Hz]")
plt.ylabel("relative phasor error")
plt.title("Accuracy versus signal frequency (fine step, dt = 1e-4 s)")
plt.legend()
plt.grid(True, which="both", linestyle=":")
plt.show()

In [ ]:
# DP-SSN wins at the carrier; far below it the fast-rotating envelope makes
# EMT-SSN the more accurate method (the crossover near 25-30 Hz).
assert dp_fine[carrier] < emt_fine[carrier]
assert dp_fine[0] > emt_fine[0]

## Sweep over signal frequency (large time step)

The same symmetric sweep at a large step makes the effect dramatic and clears the
floor. At the carrier DP-SSN is orders of magnitude more accurate; the advantage
falls off on either side as the envelope offset grows, and below the crossover
(~25-30 Hz) it drops below one, where EMT-SSN at a low absolute frequency warps
less than the fast-rotating DP envelope.

In [ ]:
dt_big = 2e-3
emt_big = [emt_ssn_error(f, dt_big, r) / abs(r) for f, r in zip(freqs, refs)]
dp_big = [dp_ssn_error(f, dt_big, r) / abs(r) for f, r in zip(freqs, refs)]

for f, ee, ed in zip(freqs, emt_big, dp_big):
    print(
        f"f_sig={f:3d} Hz (offset {f - f0:+.0f})  EMT-SSN rel={ee:.3e}  DP-SSN rel={ed:.3e}  ratio={ee / ed:.2f}"
    )

In [ ]:
plt.figure(figsize=(9, 4))
plt.semilogy(freqs, emt_big, "o-", label="EMT-SSN")
plt.semilogy(freqs, dp_big, "s-", label="DP-SSN")
plt.axvline(f0, color="0.7", linestyle=":", label="carrier")
plt.xlabel("signal frequency [Hz]")
plt.ylabel("relative phasor error")
plt.title("Accuracy versus signal frequency (large step, dt = 2e-3 s)")
plt.legend()
plt.grid(True, which="both", linestyle=":")
plt.show()

In [ ]:
advantage = [ee / ed for ee, ed in zip(emt_big, dp_big)]
print("DP-SSN advantage at the carrier:", advantage[carrier], "x")
print("DP-SSN advantage far below the carrier:", advantage[0], "x")
assert advantage[carrier] > 100  # DP-SSN dominates at the carrier
assert advantage[0] < 1  # EMT-SSN wins far below it (the crossover)